# NeSy-KG trên Flora (zero-shot skeleton): các hướng cải tiến

Baseline là **Flora** (CVPR 2026 Findings), so với hai mốc đã chạy: `none` (Flora gốc) và `compose` (cộng `β·ngữ nghĩa ghép từ primitive` vào `attuned_semantics`, β=0.3). Bốn hướng cải tiến nằm trong bốn file `zeroshot/Flora/nesy1_logic.py` ... `nesy4_seqkg.py`, dùng chung `nesy_common.py`; mỗi cell chỉ gọi `!python file.py`.

**Trước khi chạy**: phải commit và push `zeroshot/Flora` (các file `nesy*.py`, `kg/` gồm cả `*_v2seq.json`, `kg_semantics.py`, `trainer.py`, `options.py`). Notebook clone repo từ GitHub nên chỉ thấy phần đã push. Repo private thì lưu token vào Colab Secrets với tên `GH_TOKEN`.

KG chỉ được viết từ tên lớp và từ vựng primitive, không dùng dữ liệu skeleton.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, shutil, subprocess, time, re
import numpy as np, pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/BaoHuy/zeroshot'
FEAT_ARCHIVE = '/content/drive/MyDrive/BaoHuy/flora_pretrain.zip'   # feature Flora đã trích sẵn, lưu sẵn trên Drive
OUT_DIR = f'{DRIVE_DIR}/flora_nesy'                             # log + bảng kết quả
EXTRACT = '/content/flora_features'
os.makedirs(OUT_DIR, exist_ok=True)

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], check=True, stdout=open(f'{OUT_DIR}/thesis_commit.txt', 'w'))

FLORA = f'{REPO}/zeroshot/Flora'
need = ['nesy_common.py', 'nesy1_logic.py', 'nesy2_hardneg.py', 'nesy3_primtok.py', 'nesy4_seqkg.py', 'kg_semantics.py', 'kg/primitive_v2.json', 'kg/kg_rules_v2.json', 'semantics/ntu60_llm.txt', 'semantics/ntu120_llm.txt']
missing = [p for p in need if not os.path.exists(f'{FLORA}/{p}')]
assert not missing, f'Repo chưa có {missing}: commit và push zeroshot/Flora trước'
assert '--kg-mode' in open(f'{FLORA}/options.py').read(), 'options.py trên GitHub chưa có --kg-mode: chưa push bản sửa trainer/options'
for n in (60, 120):
    n_lines = len(open(f'{FLORA}/semantics/ntu{n}_llm.txt', encoding='utf-8').read().splitlines())
    assert n_lines == n, f'ntu{n}_llm.txt có {n_lines} dòng, cần {n}'
print('repo OK')

In [ ]:
!pip -q install transformers pyyaml pandas

# models/fm.py luôn import Shift-GCN, và Shift-GCN import CUDA op `shift_cuda`.
# Vì use_features=True nên Shift-GCN không bao giờ chạy, chỉ cần một module rỗng để import qua.
stub = f'{FLORA}/models/shiftgcn/Temporal_shift/cuda/shift_cuda.py'
if not os.path.exists(stub):
    with open(stub, 'w') as f:
        f.write('# stub: Shift-GCN CUDA op is never called when use_features=True\n')

!cd {FLORA} && python -c "from models.fm import MappingNet; import kg_semantics; print('import OK')"

## Giải nén feature đã trích sẵn

Feature của Flora (`star_features`, `synse_features`) đã lưu sẵn trên Drive tại `FEAT_ARCHIVE` (`MyDrive/BaoHuy/flora_pretrain.zip`), không tải lại. Cell này giải nén vào `/content` (đĩa local, đọc nhanh hơn Drive) rồi tạo symlink vào `Flora/skeleton_features/`. Sau mỗi lần restart runtime phải chạy lại cell này.

In [ ]:
assert os.path.exists(FEAT_ARCHIVE), f'Không thấy {FEAT_ARCHIVE} trên Drive'
!ls -lh {FEAT_ARCHIVE}

FEAT_NAMES = ('star_features', 'synse_features')   # star = 1s-Shift-GCN (Flora), synse = 4s-Shift-GCN (Flora†)

def find_feat(name):
    # zip nén trên macOS có thư mục __MACOSX chứa file rác ._*: bỏ qua
    hits = [h for h in glob.glob(f'{EXTRACT}/**/{name}', recursive=True) if '__MACOSX' not in h]
    return hits[0] if hits else None

if not any(find_feat(n) for n in FEAT_NAMES):
    os.makedirs(EXTRACT, exist_ok=True)
    t0 = time.time()
    subprocess.run(['unzip', '-q', '-o', FEAT_ARCHIVE, '-x', '__MACOSX/*', '*/._*', '._*', '-d', EXTRACT], check=True)
    print(f'giải nén xong trong {time.time() - t0:.0f}s')

assert any(find_feat(n) for n in FEAT_NAMES), f'Trong {FEAT_ARCHIVE} không có thư mục star_features hay synse_features'
os.makedirs(f'{FLORA}/skeleton_features', exist_ok=True)   # thư mục này nằm trong .gitignore của Flora
for name in FEAT_NAMES:
    src_dir = find_feat(name)
    link = f'{FLORA}/skeleton_features/{name}'
    if src_dir is None:
        print(f'archive không có {name}')
        continue
    if os.path.islink(link):   # link cũ có thể trỏ tới chỗ giải nén lần trước
        os.remove(link)
    os.symlink(src_dir, link)
    print(name, sorted(f for f in os.listdir(link) if not f.startswith('._')))

## Kiểm tra KG (không cần GPU)

In số primitive mỗi lớp, độ giống KG giữa các lớp, và các cặp lớp mà KG **không phân biệt được** (hàng giống hệt nhau). Các cặp đó (mặc/cởi áo, giày, kính, mũ) thành láng giềng mạnh của nhau ở chế độ `neighbors`.

In [ ]:
!cd {FLORA} && python kg_semantics.py 60
!cd {FLORA} && python kg_semantics.py 120 | head -8

## Hướng 3 (primtok): hai biến thể

Kết quả seed 1 của bốn hướng (`best`, Δ so với `compose` cùng seed). Log và `summary.csv` của từng hướng nằm trên Drive trong `OUT_DIR/<nhãn>/`.

| Hướng | 48/12 | Δ compose | 110/10 | Δ compose |
|---|---|---|---|---|
| 1 logic | 54.54 | −8.33 | 75.08 | −4.09 |
| 2 hardneg | 63.20 | +0.33 | 79.03 | −0.14 |
| **3 primtok** (trên `none`) | **64.81** | **+1.94** | 78.54 | −0.63 |
| 4 seqkg | 62.75 | −0.12 | 78.98 | −0.19 |

Hướng 1 dừng: primitive không chuyển được sang lớp unseen (độ chính xác 67%, recall `require` 39–55%). Hai cell dưới thử primtok với KG có thứ tự, và primtok cộng `compose`. Cấu hình tốt hơn sẽ chạy 3 seed trên cả 4 split (`--tasks 48 55 110 96 --seeds 1 2 3`).

In [ ]:
# primtok + KG v2seq: mỗi primitive thứ tự (seq.a>b) là một token riêng
!python {FLORA}/nesy4_seqkg.py --build-only   # dựng lại KG v2seq nếu runtime mới
!python {FLORA}/nesy3_primtok.py --kg-prims kg/primitive_v2seq.json --kg-rules kg/kg_rules_v2seq.json

In [ ]:
# primtok trên nền compose (thay vì Flora thuần)
!python {FLORA}/nesy3_primtok.py --kg-mode compose

## Chạy trên feature SynSE (Flora†)

Bảng 1 của Flora so sánh chủ yếu trên feature SynSE (4s-Shift-GCN): Flora† đạt **86.3 / 65.3 / 79.6 / 66.4** (55/5, 48/12, 110/10, 96/24), và các ablation của Flora cũng dùng feature này. `--feat synse` dùng config `configs/synse/*` và file feature trong config; run và baseline (`none`, `compose`) ghi vào `OUT_DIR/synse/`, tách khỏi run STAR. Baseline chưa có sẽ được chạy trước.

In [ ]:
!python {FLORA}/nesy3_primtok.py --kg-mode compose --feat synse --tasks 48 110 --seeds 1
